# Special Portfolios

Questi due portafogli rappresentano approcci "agnostici" al rendimento: invece di cercare di indovinare quale asset salirà di più (compito statisticamente arduo), si concentrano esclusivamente sulla gestione matematica del rischio.

Ecco a cosa servono e quando utilizzarli:

---

### 1. Portafoglio a Minima Varianza (Minimum Variance)

**Scopo: Riduzione estrema del rischio totale.**

Questo portafoglio è progettato per trovare la combinazione di asset che genera la **volatilità complessiva più bassa possibile**, tenendo conto non solo della rischiosità dei singoli titoli, ma anche di come questi si muovono insieme (correlazioni).

* **A cosa serve:** È lo strumento definitivo per la conservazione del capitale. Si colloca nel punto più a sinistra della frontiera efficiente.
* **Perché si usa:** Funziona bene quando non si hanno previsioni affidabili sui rendimenti futuri. Eliminando la variabile "rendimento atteso" dall'equazione, si riduce il rischio di errore di stima.
* **Il "Trucco" Matematico:** Sfrutta le correlazioni negative o basse. Se l'Asset A scende quando l'Asset B sale, il modello li combina per "appiattire" la curva dei rendimenti.

---

### 2. Portafoglio a Parità di Rischio (Risk Parity - Inverse Variance)

**Scopo: Diversificazione reale del rischio (non del capitale).**

In un portafoglio classico "60/40" (60% azioni, 40% obbligazioni), circa il 90% del rischio totale deriva dalle azioni perché sono molto più volatili. La Risk Parity serve a correggere questo squilibrio.

* **A cosa serve:** Assicura che ogni componente del portafoglio contribuisca in modo uguale alla rischiosità complessiva. Se un asset è molto volatile (es. Bitcoin o Azioni), riceverà un peso molto piccolo; se un asset è stabile (es. Treasury Bond), riceverà un peso maggiore.
* **Perché si usa:** Per ottenere rendimenti stabili in diversi scenari economici. È la filosofia dietro il fondo "All Weather" di Ray Dalio: non importa se il mercato è in crescita o in recessione, il portafoglio è bilanciato per non essere abbattuto da un singolo crash settoriale.
* **Il "Trucco" Matematico:** Il peso $w_i$ dell'asset $i$ è calcolato come:

$$w_i = \frac{1/\sigma_i^2}{\sum_{j=1}^{n} 1/\sigma_j^2}$$



Dove $\sigma_i^2$ è la varianza dell'asset.

---

### Tabella Comparativa

| Caratteristica | Minima Varianza | Risk Parity |
| --- | --- | --- |
| **Obiettivo Primario** | Minimizzare la volatilità totale del portafoglio. | Distribuire il rischio equamente tra gli asset. |
| **Input Chiave** | Matrice di Covarianza (Volatilità + Correlazioni). | Volatilità dei singoli asset (nella versione base). |
| **Comportamento** | Può concentrarsi molto su pochi asset molto sicuri. | Tende a essere più diversificato tra tutte le asset class. |
| **Scenario Ideale** | Mercati altamente instabili e imprevedibili. | Strategia di lungo termine per ogni ciclo economico. |

### Quale scegliere?

* Usa la **Minima Varianza** se la tua priorità assoluta è non vedere grandi oscillazioni nel valore del conto, accettando rendimenti potenzialmente più bassi.
* Usa la **Risk Parity** se vuoi un sistema robusto che non dipenda troppo dall'andamento di una singola classe di attività (come l'azionario).

## Potremmo cambiare i pesi di questi due portafogli aggiungendo un vincolo di **Target Volatility** (es. forzare il portafoglio ad avere esattamente il 10% di volatilità annua)

1. Minimum Variance Portfolio L'obiettivo matematico è minimizzare la funzione obiettivo $f(w) = w^T \Sigma w$. A differenza della Mean-Variance classica, questa non richiede stime sui rendimenti attesi ($E[r]$), eliminando la principale fonte di errore di stima ("garbage in, garbage out"). È la strategia che solitamente performa meglio in contesti di alta incertezza.

2. Risk Parity (Inverse Variance)Mentre l'ottimizzazione di varianza minima cerca la combinazione che riduce il rischio totale, la Risk Parity mira a eguagliare il contributo al rischio di ogni asset. La versione presentata (inversa della varianza) è la soluzione analitica quando le correlazioni tra asset sono assunte come costanti o nulle.Considerazioni sul Risk BudgetingPer rendere questo sistema pronto per una produzione di tipo "Asset Allocator professionale", occorrerebbe integrare:Shrinkage della Covarianza: Utilizzare lo stimatore di Ledoit-Wolf invece della covarianza campionaria per gestire il rumore nei dati storici.Target Volatility: Scalare i pesi per mantenere una volatilità di portafoglio costante (es. 10% annuo).

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

class SpecialPortfolios:
    def __init__(self, returns):
        """
        returns: DataFrame dei rendimenti storici degli asset
        """
        self.returns = returns
        self.cov_matrix = returns.cov() * 12  # Covarianza annualizzata
        self.n_assets = len(returns.columns)

    def equal_weights(self):
        """Strategia 1/N"""
        return np.ones(self.n_assets) / self.n_assets

    def risk_parity_variance(self):
        """
        Pesi inversamente proporzionali alla varianza (Risk Parity Vanilla).
        """
        variances = np.diag(self.cov_matrix)
        inv_variances = 1.0 / variances
        return inv_variances / np.sum(inv_variances)

    def minimum_variance(self, short_limit=-1.0):
        """
        Ottimizzazione per il portafoglio a varianza minima.
        Vincolo: somma dei pesi = 1.
        Bounds: short selling limitato a -100% per asset.
        """
        def portfolio_variance(weights):
            return weights.T @ self.cov_matrix @ weights

        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        bounds = tuple((short_limit, 1.0) for _ in range(self.n_assets))
        initial_guess = self.equal_weights()

        result = minimize(portfolio_variance, initial_guess,
                          method='SLSQP', bounds=bounds, constraints=constraints)

        if result.success:
            return result.x
        else:
            raise ValueError("L'ottimizzazione non ha prodotto una soluzione.")

    def get_comparison_table(self):
        """Genera un riepilogo dei pesi per ogni strategia"""
        weights_df = pd.DataFrame(index=self.returns.columns)
        weights_df['Equal Weight'] = self.equal_weights()
        weights_df['Risk Parity (Var)'] = self.risk_parity_variance()
        weights_df['Min Variance'] = self.minimum_variance()
        return weights_df

# Esempio d'uso con dati sintetici
if __name__ == "__main__":
    # Generazione dati random: 4 asset (T-Bond, Corp Bond, Stock US, Stock Int)
    np.random.seed(42)
    data = pd.DataFrame(np.random.normal(0.005, 0.04, (120, 4)),
                        columns=['UST', 'US_Corp', 'S&P500', 'MSCI_EAFE'])

    portfolios = SpecialPortfolios(data)
    results = portfolios.get_comparison_table()
    print(results.round(4))

           Equal Weight  Risk Parity (Var)  Min Variance
UST                0.25             0.2820        0.3162
US_Corp            0.25             0.2626        0.2466
S&P500             0.25             0.2073        0.2186
MSCI_EAFE          0.25             0.2481        0.2187
